# Profiling regions

> **Install for this page:** `pip install scope-profiler` (base install only).

A **region** is the fundamental unit of measurement in scope-profiler.
Every time a region is entered and exited, the profiler records a
start/end timestamp pair and increments the call counter.

In [ ]:
import os
import sys
import tempfile
from pathlib import Path

# Profile into a scratch directory so nothing lands in the working directory,
# and make the `scope-profiler` command of this environment available.
WORKDIR = Path(tempfile.mkdtemp(prefix="scope-profiler-guide-"))
os.chdir(WORKDIR)
os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ["PATH"]

import time

import numpy as np

from scope_profiler import ProfileManager

Regions only record while profiling is configured, so the cells on this page
share one profiling session, started here and written at the end. The output
goes to the scratch directory.

In [ ]:
run = ProfileManager.setup(file_path="regions.h5")

## Creating regions

### Decorator — `@ProfileManager.profile`

Ideal for wrapping entire functions:

In [ ]:
@ProfileManager.profile("solver")
def solve(A, b):
    return np.linalg.solve(A, b)


A, b = np.eye(3), np.ones(3)
print(solve(A, b))
print(ProfileManager.get_region("solver").num_calls)

Without an explicit name the function name is used:

In [ ]:
@ProfileManager.profile
def solve(A, b):
    return np.linalg.solve(A, b)


# region name: "solve"
print(solve(A, b))
print("solve" in ProfileManager.recorded_regions())

You can recursively profile nested Python function calls from a decorated
entrypoint:

In [ ]:
def run_pipeline(A, b):
    return sum(b)


@ProfileManager.profile("solve", recursive=True)
def solve(A, b):
    return run_pipeline(A, b)


print(solve(A, b))

Nested calls are stored as additional regions using fully-qualified names
such as `my_module.run_pipeline`.

In [ ]:
[name for name in ProfileManager.recorded_regions() if "run_pipeline" in name]

### Context manager — `ProfileManager.region()`

Ideal for profiling a section of code inside a function:

In [ ]:
def compute_rhs(state):
    return 2.0 * state


def time_step(state, dt):
    with ProfileManager.region("rhs_evaluation"):
        rhs = compute_rhs(state)

    with ProfileManager.region("state_update"):
        state += dt * rhs


time_step(np.ones(3), 0.1)

`profile_region()` is the original name for this method and still works —
the two are the same object, so existing instrumentation needs no change.

In [ ]:
# Bound methods are created on access, so compare the underlying functions.
ProfileManager.profile_region.__func__ is ProfileManager.region.__func__

### Mixing both

Decorators and context managers can be mixed freely. They share the
same region registry, so using the same name in both places accumulates
into a single region:

In [ ]:
def process(chunk):
    time.sleep(0.001)


@ProfileManager.profile("compute")
def compute_batch(data):
    for chunk in data:
        with ProfileManager.region("compute"):
            process(chunk)


# Both the decorator and the context manager contribute
# to the same "compute" region.
compute_batch([1, 2, 3])
print(ProfileManager.get_region("compute").num_calls)

One call of the decorated function plus three entries of the context manager
make four calls of the single `compute` region.

## Region identity

Regions are identified by **name** (a string). The first call to
`region("foo")` creates the region; subsequent calls with the same name return
the existing instance. This means:

- The same region can be entered from multiple call sites.
- Call counts and timestamps accumulate across all sites.
- Regions are scoped to the current `ProfileManager.setup()` session.
  Calling `setup()` again clears all regions.

In [ ]:
ProfileManager.region("foo") is ProfileManager.region("foo")

## Nesting

Regions can be nested arbitrarily:

In [ ]:
def work():
    time.sleep(0.001)


with ProfileManager.region("outer"):
    with ProfileManager.region("inner"):
        work()

Each region independently records its own timestamps. The library does
**not** compute parent-child relationships — that can be done in
post-processing by comparing the recorded intervals.

## Accessing region data at runtime

Before calling `finalize()`, you can inspect in-memory data:

In [ ]:
region = ProfileManager.get_region("solver")

# Number of completed calls
print(region.num_calls)

# Durations of buffered (not yet flushed) calls, in nanoseconds
durations = region.get_durations_numpy()
print(durations.shape, durations.dtype)

All registered regions:

In [ ]:
for name, region in ProfileManager.get_all_regions().items():
    print(f"{name}: {region.num_calls} calls")

Finalizing writes the file. A following `setup()` starts a new session, which
clears all regions:

In [ ]:
ProfileManager.finalize(verbose=False)
ProfileManager.setup(file_path="regions.h5")
print(ProfileManager.recorded_regions())
ProfileManager.finalize(verbose=False)

## Tagging regions

Both `@ProfileManager.profile` and `ProfileManager.region()` take `tags`, labels
stored with the region. `setup(region_tags=...)` then records only the regions
that carry one of the listed tags, which gives a cheap default profile of a
subset of the instrumented code:

In [ ]:
@ProfileManager.profile("assemble", tags=["core"])
def assemble(n):
    return np.eye(n)


@ProfileManager.profile("diagnostics", tags=["detail"])
def diagnostics(matrix):
    return float(np.trace(matrix))


def step(n):
    matrix = assemble(n)
    with ProfileManager.region("residual", tags=["detail"]):
        diagnostics(matrix)


ProfileManager.setup(file_path="tagged.h5", region_tags=["core"])
step(3)
print(ProfileManager.finalize(verbose=False, return_results=True).region_names)

Passing `region_tags=None` (the default) goes back to the full profile; the
decorated functions are rebound by `setup()`, so nothing has to be redefined:

In [ ]:
ProfileManager.setup(file_path="tagged.h5", region_tags=None)
step(3)
results = ProfileManager.finalize(verbose=False, return_results=True)
print(sorted(results.region_names))
print(results["diagnostics"].tags)

Inside `ProfileManager.session()`, the session's own `scope_profiler.session`
region is always recorded, so the total run time stays visible however the
regions are filtered.